# Clean Train and Test Datasets Creation

In [ ]:
import os
import shutil
import random
from pathlib import Path
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from data_loader.dataset import EuroSatDataset

In [ ]:
def create_and_save_datasets(
    data_path,
    save_path,
    train_ratio=0.8,
    seed=42
):
    random.seed(seed)

    data_path = Path(data_path)
    save_path = Path(save_path)

    train_dir = save_path / "train_clean"
    test_dir = save_path / "test_clean"

    # Remove existing directories if they exist
    if train_dir.exists():
        print(f"Removing existing directory: {train_dir}")
        shutil.rmtree(train_dir)
    
    if test_dir.exists():
        print(f"Removing existing directory: {test_dir}")
        shutil.rmtree(test_dir)

    classes = sorted([d for d in data_path.iterdir() if d.is_dir()])

    print(f"Found {len(classes)} classes")

    for class_dir in classes:
        class_name = class_dir.name

        # Créer les dossiers de sortie
        (train_dir / class_name).mkdir(exist_ok=True)
        (test_dir / class_name).mkdir(exist_ok=True)

        # Lister les images
        images = sorted(
            [img for img in list(class_dir.glob("*"))
            if img.suffix.lower() in [".jpg", ".png", ".jpeg"]]
        )

        random.Random(seed).sample(images, len(images))

        n_train = int(len(images) * train_ratio)

        train_images = images[:n_train]
        test_images = images[n_train:]

        # Copier les fichiers
        for img_path in train_images:
            shutil.copy(img_path, train_dir / class_name / img_path.name)

        for img_path in test_images:
            shutil.copy(img_path, test_dir / class_name / img_path.name)

        print(
            f"Class {class_name}: "
            f"{len(train_images)} train / {len(test_images)} test"
        )

    print("\nDataset split completed.")
    print(f"Train directory: {train_dir}")
    print(f"Test directory: {test_dir}")


create_and_save_datasets(data_path="data/EuroSAT_RGB", save_path="datasets/EuroSAT_RGB", train_ratio=0.8, seed=42)

Found 10 classes
Class AnnualCrop: 2400 train / 600 test
Class Forest: 2400 train / 600 test
Class HerbaceousVegetation: 2400 train / 600 test
Class Highway: 2000 train / 500 test
Class Industrial: 2000 train / 500 test
Class Pasture: 1600 train / 400 test
Class PermanentCrop: 2000 train / 500 test
Class Residential: 2400 train / 600 test
Class River: 2000 train / 500 test
Class SeaLake: 2400 train / 600 test

Dataset split completed.
Train directory: datasets/EuroSAT_RGB/train_clean
Test directory: datasets/EuroSAT_RGB/test_clean


In [ ]:
def compute_stats():
    train_path = "./datasets/EuroSAT_RGB/train_clean"
    
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    
    # Loads only train set
    dataset = EuroSatDataset(
        root_dir=train_path,
        transform=transform,
        train=True
    )
    
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    
    mean = 0.
    std = 0.
    nb_samples = 0.
    
    for data, _ in dataloader:
        # data: [batch, 3, H, W]
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)
        
        mean += data.mean(2).sum(0)
        std += data.std(2).sum(0)
        nb_samples += batch_samples
    
    mean /= nb_samples
    std /= nb_samples
    
    print(f"MEAN = {mean.tolist()}")
    print(f"STD = {std.tolist()}")
    
    return mean.tolist(), std.tolist()


def save_statistics_to_config(mean, std, config_file_path="config.py"):
    """
    Saves statistics into config file.
    
    Args:
        mean: Mean list [R, G, B]
        std: Standard errors list [R, G, B]
        config_file_path: Path to config.py
    """
    
    with open(config_file_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        if line.strip().startswith("MEAN ="):
            new_lines.append(f"MEAN = {mean}  # Computed on train set\n")
        elif line.strip().startswith("STD ="):
            new_lines.append(f"STD = {std}  # Computed on train set\n")
        else:
            new_lines.append(line)
    
    # Écrire le fichier mis à jour
    with open(config_file_path, 'w') as f:
        f.writelines(new_lines)
    
    print(f"Statistiques sauvegardées dans {config_file_path}")


mean, std = compute_stats()
save_statistics_to_config(mean, std, config_file_path="config.py")

# Baseline Model Creation

In [6]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/baseline",
    "--save-plots-path", "outputs/plots/baseline_clean",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Train Loss: 1.2226 | Train Acc: 56.34%
Val Loss: 0.9438 | Val Acc: 64.60%
Saved best model with val_acc: 64.60%

Epoch 2/50
--------------------------------------------------


Train Loss: 0.8091 | Train Acc: 71.34%
Val Loss: 0.5906 | Val Acc: 79.31%
Saved best model with val_acc: 79.31%

Epoch 3/50
--------------------------------------------------


Train Loss: 0.6351 | Train Acc: 78.04%
Val Loss: 0.5033 | Val Acc: 82.98%
Saved best model with val_acc: 82.98%

Epoch 4/50
--------------------------------------------------


Train Loss: 0.5058 | Train Acc: 82.51%
Val Loss: 0.4516 | Val Acc: 84.18%
Saved best model with val_acc: 84.18%

Epoch 5/50
--------------------------------------------------


...

Train Loss: 0.0225 | Train Acc: 99.20%
Val Loss: 0.0736 | Val Acc: 98.02%

Epoch 47/50
--------------------------------------------------


Train Loss: 0.0155 | Train Acc: 99.47%
Val Loss: 0.0651 | Val Acc: 97.93%

Epoch 48/50
--------------------------------------------------


Train Loss: 0.0138 | Train Acc: 99.54%
Val Loss: 0.0631 | Val Acc: 98.04%

Epoch 49/50
--------------------------------------------------


Train Loss: 0.0118 | Train Acc: 99.58%
Val Loss: 0.0602 | Val Acc: 98.24%

Epoch 50/50
--------------------------------------------------


Train Loss: 0.0115 | Train Acc: 99.64%
Val Loss: 0.0622 | Val Acc: 98.09%

Training completed! Best validation accuracy: 98.26%
Training history plot saved to: outputs/plots/baseline_clean/resnet18_training_history.png
Training history plot saved to: outputs/plots/baseline_clean/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/baseline_clean/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/baseline_clean/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 98.17%
✓ Evaluation logs saved to: outputs/plots/baseline_clean
  - Human-readable: outputs/plots/baseline_clean/evaluation_20260116_175759.log
  - JSON details: outputs/plots/baseline_clean/evaluation_20260116_175759.json
  - Simple log: outputs/plots/baseline_clean/evaluation.log
Confusion matrix saved to: outputs/plots/baseline_clean/confusion_matrix_normalized.png

Classification Report:
                      prec

# Adversarial Test Dataset Creation

In [8]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm
import json

from models import ResNet18
from data_loader.dataset import EuroSatDataset
from attacks.pgd import PGD
from config import BATCH_SIZE, DEVICE, MEAN, STD, SEED

In [9]:
def load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model
    model = ResNet18().to(device)

    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"Model loaded from {model_path}")
    else:
        raise FileNotFoundError(f"Model not found at {model_path}")

    model.eval()

    # Load clean test dataset
    dataset = EuroSatDataset(
        root_dir=test_clean_path,
        train=False
    )

    class_names = dataset.classes

    test_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    # Prepare save folders
    os.makedirs(save_adv_path, exist_ok=True)
    for cls in class_names:
        os.makedirs(os.path.join(save_adv_path, cls), exist_ok=True)

    # PGD attack
    epsilon = torch.tensor([epsilon_pixel for _ in range(3)]).view(1,3,1,1)
    alpha   = torch.tensor([alpha_pixel for _ in range(3)]).view(1,3,1,1)

    pgd = PGD(
        model=model,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations,
        random_start=True,
        device=device,
        seed=SEED,
    )

    # Generate & save adversarial images
    img_idx = 0

    for images, labels in tqdm(test_loader, desc="Generating adversarial test set"):
        images = images.to(device)
        labels = labels.to(device)

        with torch.enable_grad():
            adv_images = pgd.attack(images, labels)

        for i in range(adv_images.size(0)):
            label = labels[i].item()
            class_name = class_names[label]

            save_path = os.path.join(
                save_adv_path,
                class_name,
                f"img_{img_idx}.png"
            )

            save_image(adv_images[i], save_path)
            img_idx += 1

    print(f"\nAdversarial test dataset saved to: {save_adv_path}")

    
    attack_config = {
        "attack": "PGD",
        "epsilon": epsilon.tolist() if torch.is_tensor(epsilon) else epsilon,
        "alpha": alpha.tolist() if torch.is_tensor(alpha) else alpha,
        "iterations": iterations,
        "random_start": True,
        "seed": SEED,
        "normalization": {
            "mean": MEAN,
            "std": STD
        }
    }

    os.makedirs("attacks/configs", exist_ok=True)
    with open(os.path.join("attacks/configs", "attack_config.json"), "w") as f:
        json.dump(attack_config, f, indent=4)

    return None

load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/baseline/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5
)

Using device: cuda
Model loaded from outputs/models/baseline/best_model.pth
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

         [[0.0040]],

         [[0.0040]]]]) might be too large for ε=tensor([[[[0.0200]],

         [[0.0200]],

         [[0.0200]]]]), iterations=5


Generating adversarial test set:   0%|          | 0/169 [00:00<?, ?it/s]

Generating adversarial test set: 100%|██████████| 169/169 [01:45<00:00,  1.60it/s]


Adversarial test dataset saved to: datasets/EuroSAT_RGB/test_pgd_eps002


In [10]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/baseline_pgd_eps002",
    "--save-model-path", "outputs/model/baseline",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...


Test Accuracy: 20.28%


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-pa

✓ Evaluation logs saved to: outputs/plots/baseline_pgd_eps002
  - Human-readable: outputs/plots/baseline_pgd_eps002/evaluation_20260116_181006.log
  - JSON details: outputs/plots/baseline_pgd_eps002/evaluation_20260116_181006.json
  - Simple log: outputs/plots/baseline_pgd_eps002/evaluation.log
Confusion matrix saved to: outputs/plots/baseline_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.00      0.00      0.00       600
              Forest       0.27      0.98      0.43       600
HerbaceousVegetation       0.11      0.36      0.17       600
             Highway       0.06      0.00      0.01       500
          Industrial       0.00      0.00      0.00       500
             Pasture       0.00      0.00      0.00       400
       PermanentCrop       0.24      0.57      0.34       500
         Residential       0.00      0.00      0.00       600
               River       0

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/baseline_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 0.00%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


# Madry

## epsilon = 0.03, alpha = 0.008, pgd_steps = 7

In [12]:
import sys
import json

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "10",
    "--lr", "0.001",
    "--batch-size", "32",
    "--madry", json.dumps({
        "epsilon": 0.03,
        "alpha": 0.008,
        "pgd_steps": 7
    }),
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/madry_eps003",
    "--save-plots-path", "outputs/plots/madry_eps003",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Training:   0%|          | 0/473 [00:00<?, ?it/s]

Train Loss: 2.0035 | Train Acc: 20.12%
Val Loss: 2.2815 | Val Acc: 17.55%
Saved best model with val_acc: 17.55%

Epoch 2/50
--------------------------------------------------


Train Loss: 1.7831 | Train Acc: 27.94%
Val Loss: 2.0411 | Val Acc: 25.68%
Saved best model with val_acc: 25.68%

Epoch 3/50
--------------------------------------------------


Train Loss: 1.6862 | Train Acc: 31.20%
Val Loss: 2.2358 | Val Acc: 21.71%

Epoch 4/50
--------------------------------------------------


...

Train Loss: 1.0945 | Train Acc: 56.62%
Val Loss: 5.8325 | Val Acc: 22.39%

Early stopping triggered after 26 epochs with no improvement.

Training completed! Best validation accuracy: 34.78%
Training history plot saved to: outputs/plots/madry_eps003/resnet18_training_history.png
Training history plot saved to: outputs/plots/madry_eps003/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/madry_eps003/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/madry_eps003/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 33.78%


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-pa

✓ Evaluation logs saved to: outputs/plots/madry_eps003
  - Human-readable: outputs/plots/madry_eps003/evaluation_20260116_203830.log
  - JSON details: outputs/plots/madry_eps003/evaluation_20260116_203830.json
  - Simple log: outputs/plots/madry_eps003/evaluation.log
Confusion matrix saved to: outputs/plots/madry_eps003/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.76      0.45      0.57       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetation       0.21      0.02      0.04       600
             Highway       0.14      0.00      0.00       500
          Industrial       0.51      0.77      0.61       500
             Pasture       0.33      0.50      0.39       400
       PermanentCrop       0.33      0.30      0.31       500
         Residential       0.26      0.89      0.41       600
               River       0.41      0.06      0.10       500
 

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps003/sample_predictions.png
Batch accuracy on 16 samples: 43.75%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


In [13]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/madry_eps003/test_pgd_eps002",
    "--save-model-path", "outputs/models/madry_eps003",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Loaded best model for evaluation


Test Accuracy: 28.06%


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-pa

✓ Evaluation logs saved to: outputs/plots/madry_eps003/test_pgd_eps002
  - Human-readable: outputs/plots/madry_eps003/test_pgd_eps002/evaluation_20260116_204032.log
  - JSON details: outputs/plots/madry_eps003/test_pgd_eps002/evaluation_20260116_204032.json
  - Simple log: outputs/plots/madry_eps003/test_pgd_eps002/evaluation.log
Confusion matrix saved to: outputs/plots/madry_eps003/test_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.49      0.67      0.56       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetation       0.12      0.01      0.01       600
             Highway       0.00      0.00      0.00       500
          Industrial       0.19      0.35      0.25       500
             Pasture       0.00      0.00      0.00       400
       PermanentCrop       0.22      0.59      0.32       500
         Residential       0.17      0.34    

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps003/test_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 68.75%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


## epsilon = 0.01, alpha = 0.002, pgd_steps = 7

In [14]:
import sys
import json

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "64",
    "--madry", json.dumps({
        "epsilon": 0.01,
        "alpha": 0.002,
        "pgd_steps": 7
    }),
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/madry_eps001",
    "--save-plots-path", "outputs/plots/madry_eps001",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Training:   0%|          | 0/237 [00:00<?, ?it/s]

Train Loss: 1.8692 | Train Acc: 23.51%
Val Loss: 2.8098 | Val Acc: 21.67%
Saved best model with val_acc: 21.67%

Epoch 2/50
--------------------------------------------------


Train Loss: 1.6013 | Train Acc: 36.68%
Val Loss: 3.0122 | Val Acc: 30.25%
Saved best model with val_acc: 30.25%

Epoch 3/50
--------------------------------------------------


Train Loss: 1.4905 | Train Acc: 40.09%
Val Loss: 3.8835 | Val Acc: 31.48%
Saved best model with val_acc: 31.48%

Epoch 4/50
--------------------------------------------------


...

Train Loss: 0.9809 | Train Acc: 60.07%
Val Loss: 30.7524 | Val Acc: 27.31%

Epoch 21/50
--------------------------------------------------


Train Loss: 0.9625 | Train Acc: 60.77%
Val Loss: 34.1842 | Val Acc: 29.37%

Epoch 22/50
--------------------------------------------------


Train Loss: 0.9599 | Train Acc: 60.95%
Val Loss: 38.6891 | Val Acc: 27.58%

Epoch 23/50
--------------------------------------------------


Train Loss: 0.9576 | Train Acc: 61.07%
Val Loss: 36.8692 | Val Acc: 28.36%

Epoch 24/50
--------------------------------------------------


Train Loss: 0.9499 | Train Acc: 61.00%
Val Loss: 38.5458 | Val Acc: 29.20%

Early stopping triggered after 24 epochs with no improvement.

Training completed! Best validation accuracy: 35.79%
Training history plot saved to: outputs/plots/madry_eps001/resnet18_training_history.png
Training history plot saved to: outputs/plots/madry_eps001/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/madry_eps001/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/madry_eps001/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 36.19%
✓ Evaluation logs saved to: outputs/plots/madry_eps001
  - Human-readable: outputs/plots/madry_eps001/evaluation_20260116_230529.log
  - JSON details: outputs/plots/madry_eps001/evaluation_20260116_230529.json
  - Simple log: outputs/plots/madry_eps001/evaluation.log
Confusion matrix saved to: outputs/plots/madry_eps001/confusion_matrix_normalized.png

Cla

In [15]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/madry_eps001/test_pgd_eps002",
    "--save-model-path", "outputs/models/madry_eps001",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Loaded best model for evaluation


Test Accuracy: 24.43%


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-pa

✓ Evaluation logs saved to: outputs/plots/madry_eps001/test_pgd_eps002
  - Human-readable: outputs/plots/madry_eps001/test_pgd_eps002/evaluation_20260116_231037.log
  - JSON details: outputs/plots/madry_eps001/test_pgd_eps002/evaluation_20260116_231037.json
  - Simple log: outputs/plots/madry_eps001/test_pgd_eps002/evaluation.log
Confusion matrix saved to: outputs/plots/madry_eps001/test_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.52      0.62      0.56       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetation       0.00      0.00      0.00       600
             Highway       0.28      0.47      0.35       500
          Industrial       0.25      0.88      0.39       500
             Pasture       0.20      0.01      0.01       400
       PermanentCrop       0.52      0.24      0.33       500
         Residential       0.04      0.10    

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps001/test_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 50.00%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


# Mixed Train Dataset Creation